## Forecasts sept 2026 - août 2027
1. Create a new folder for the month e.g. `sep_26_forecasts`
2. Download the following files from last run MLflow: 
    a. from daily run : "prophet_daily_pricing_<run>_monthly_forecast.xlsx" and "prophet_daily_pricing_20260601_060521_backtest_forecast.xlsx"
    b. from weekly run:  "prophet_weekly_pricing_<run>_monthly_forecast.xlsx" and "prophet_weekly_pricing_20260601_060601_backtest_forecast.xlsx"
   
    and put them in the folder 
3. copy the excel `Forecast_DAF_2026.xlsx` containing the predictions of the DAF
4. FILL in the first cell with the current run variables 
5. Run the cells. 

DO NOT COMMIT THE DATA !!
| Note: you might get occasional errors, you gotta fix them yourself like a big boy/girl 


In [ ]:
## FIRST RUN `gcloud auth application-default login`

## Then FILL ME THEN RUN CELLS:
first_month_prediction = "" # e.g: "2026.04.01"
last_month_of_prediction = "" # e.g: "2027.03.01"
first_month_of_backtest=""
last_month_of_backtest=""

daily_monthly_predictions_filename="" # e.g: "prophet_daily_pricing_<run>_monthly_forecast.xlsx"
weekly_monthly_predictions_filename="" # e.g: "prophet_weekly_pricing_<run>_monthly_forecast.xlsx"

daily_backtest_predictions_filename = "" #e.g.: "prophet_daily_pricing_20260601_060521_backtest_forecast.xlsx"
weekly_backtest_predictions_filename = "" #e.g.: "prophet_weekly_pricing_20260601_060601_backtest_forecast.xlsx"


In [ ]:
import pandas as pd

preds_daily = pd.read_excel(daily_monthly_predictions_filename) 
preds_weekly = pd.read_excel(weekly_monthly_predictions_filename) 
preds_DAF = pd.read_excel("Forecast_DAF_2026.xlsx")


preds_merge = pd.merge(preds_daily, preds_weekly, on="ds", suffixes=("_daily", "_weekly"))
preds_DAF.mois = pd.to_datetime(preds_DAF.mois.str.replace(" ", ""), format='%Y-%m').dt.to_period('M')
preds_merge.ds = pd.to_datetime(preds_merge.ds).dt.to_period('M')

preds_merge = pd.merge(preds_merge, preds_DAF, left_on="ds", right_on="mois", how="left")
preds_merge.drop(columns=["mois"], inplace=True)

preds_merge.rename(columns={"total_pricing_daily": "forecast_prophet_daily", "total_pricing_weekly": "forecast_prophet_weekly", "DAF": "forecast_DAF", "ds":"mois"}, inplace=True)
preds_merge

In [ ]:
preds_merge.to_excel("preds_merge.xlsx", index=False)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

plt.figure(figsize=(12, 6))
plt.plot(preds_merge.mois.astype(str), preds_merge.forecast_prophet_daily, label="Prophet Daily", marker='o')
plt.plot(preds_merge.mois.astype(str), preds_merge.forecast_prophet_weekly, label="Prophet Weekly", marker='o')
plt.plot(preds_merge.mois.astype(str), preds_merge.scénario1, label="DAF scénario1", marker='o')
plt.plot(preds_merge.mois.astype(str), preds_merge.scénario2, label="DAF scénario2", marker='o')

plt.xlabel("Mois")
plt.ylabel("Pricing en euros")
plt.title(f"Comparaison de Forecast DAF et Prophet sur la période {first_month_prediction} - {last_month_of_prediction}")

ax = plt.gca()
ax.ticklabel_format(style="plain", axis="y")  # supprime 1e7
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f"{x:,.0f}".replace(",", " ")))

plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig(f"forecast_{first_month_prediction}_{last_month_of_prediction}.png")
plt.show()

In [ ]:
GROUP_SIZE = 3
MONTH_ABBR = ["janv", "févr", "mars", "avr", "mai", "juin", "juil", "août", "sept", "oct", "nov", "déc"]


def label_period(months):
    start, end = months.min(), months.max()
    start_label = MONTH_ABBR[start.month - 1]
    end_label = MONTH_ABBR[end.month - 1]
    if start.year == end.year:
        return f"{start_label}-{end_label} {start.year}"
    return f"{start_label} {start.year}-{end_label} {end.year}"


groups = pd.Series(range(len(preds_merge)), index=preds_merge.index) // GROUP_SIZE
period_labels = preds_merge.mois.groupby(groups).agg(label_period)

preds_grouped = preds_merge.groupby(groups).sum(numeric_only=True)
preds_grouped.index = pd.Index(period_labels, name="période")
preds_grouped


## Backtest metrics

In [ ]:
## backtest forecats dataframes

df_daily_backtest = pd.read_excel(daily_backtest_predictions_filename)
df_weekly_backtest = pd.read_excel(weekly_backtest_predictions_filename)

## aggrehate to monthly
df_daily_backtest["ds"] = pd.to_datetime(df_daily_backtest["ds"]).dt.to_period('M')
df_weekly_backtest["ds"] = pd.to_datetime(df_weekly_backtest["ds"]).dt.to_period('M')

df_daily_backtest_monthly = df_daily_backtest.groupby("ds")["yhat"].sum().reset_index()
df_weekly_backtest_monthly = df_weekly_backtest.groupby("ds")["yhat"].sum().reset_index()

In [ ]:
df_weekly_backtest_monthly

In [ ]:
## GET monthly pricing from bigquery
import pandas as pd
real_pricing = pd.read_gbq("""
SELECT 
DATE_TRUNC(pricing_day, MONTH) AS pricing_month,
    SUM(total_pricing) AS real_monthly_total
FROM `passculture-data-prod.ml_finance_prod.daily_pricing`
GROUP BY pricing_month
ORDER BY pricing_month;  """, project_id="passculture-data-prod")

real_pricing.pricing_month = pd.to_datetime(real_pricing.pricing_month).dt.to_period('M')
real_pricing = pd.merge(real_pricing, preds_DAF, left_on="pricing_month", right_on="mois", how="inner")
real_pricing = pd.merge(real_pricing, df_daily_backtest_monthly, left_on="pricing_month", right_on="ds", how="inner", suffixes=("_DAF", "_daily_backtest"))
real_pricing = pd.merge(real_pricing, df_weekly_backtest_monthly, left_on="pricing_month", right_on="ds", how="inner", suffixes=("", "_weekly_backtest"))
real_pricing.rename(columns={"yhat": "daily", "yhat_weekly_backtest": "weekly"}, inplace=True)
real_pricing = real_pricing[["pricing_month", "real_monthly_total", "scénario1", "scénario2", "daily", "weekly"]]
real_pricing

In [ ]:
real_pricing.sum(numeric_only=True)

In [ ]:
print(f"Metrics on backtest set from {first_month_of_backtest} to {last_month_of_backtest}")
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error
print("Scénario 1   :")
mae = mean_absolute_error(real_pricing.real_monthly_total, real_pricing.scénario1)
rmse = root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.scénario1)
mape = mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.scénario1)
print(f"MAE: {mae:.2f} euros")
print(f"RMSE: {rmse:.2f} euros")
print(f"MAPE: {mape:.2%}")

print("Scénario 2   :")
mae = mean_absolute_error(real_pricing.real_monthly_total, real_pricing.scénario2)
rmse = root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.scénario2)
mape = mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.scénario2)
print(f"MAE: {mae:.2f} euros")
print(f"RMSE: {rmse:.2f} euros")
print(f"MAPE: {mape:.2%}")

print("Prophet Daily   :")
mae = mean_absolute_error(real_pricing.real_monthly_total, real_pricing.daily)
rmse = root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.daily)
mape = mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.daily)
print(f"MAE: {mae:.2f} euros")
print(f"RMSE: {rmse:.2f} euros")
print(f"MAPE: {mape:.2%}")

print("Prophet Weekly   :")
mae = mean_absolute_error(real_pricing.real_monthly_total, real_pricing.weekly)
rmse = root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.weekly)
mape = mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.weekly)
print(f"MAE: {mae:.2f} euros")
print(f"RMSE: {rmse:.2f} euros")
print(f"MAPE: {mape:.2%}")

In [ ]:
# dataframe with all metrics
print(f"Metrics on backtest set from {first_month_of_backtest} to {last_month_of_backtest}")

metrics_df = pd.DataFrame({
    "Model": ["DAF scénario1", "DAF scénario2", "Prophet Daily", "Prophet Weekly"],
    "MAE": [mean_absolute_error(real_pricing.real_monthly_total, real_pricing.scénario1),
            mean_absolute_error(real_pricing.real_monthly_total, real_pricing.scénario2),
            mean_absolute_error(real_pricing.real_monthly_total, real_pricing.daily),
            mean_absolute_error(real_pricing.real_monthly_total, real_pricing.weekly)],
    "RMSE": [root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.scénario1),
             root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.scénario2),
             root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.daily),
             root_mean_squared_error(real_pricing.real_monthly_total, real_pricing.weekly)],
    "MAPE": [mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.scénario1)*100,
             mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.scénario2)*100,
             mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.daily)*100,
             mean_absolute_percentage_error(real_pricing.real_monthly_total, real_pricing.weekly)*100]
})
#display with 2 decimals for MAE and RMSE, and percentage for MAPE
metrics_df = metrics_df.round({"MAE": 2, "RMSE": 2, "MAPE": 2})
metrics_df["MAPE"] = metrics_df["MAPE"].astype(str) + " %"
metrics_df

In [ ]:
df_all = pd.concat([real_pricing, preds_merge], axis=0, ignore_index=True)
df_all["pricing_month"] = df_all["pricing_month"].astype("period[M]").combine_first(df_all["mois"])
df_all.drop(columns=["mois"], inplace=True)
df_all.rename(columns={"real_monthly_total": "pricing réel effectué", "scénario1": "scenario1 DAF", "scénario2": "scenario2 DAF", "daily": "Modéle Prophet Daily", "weekly": "Modéle Prophet Weekly"}, inplace=True)

# backtest et forecast décrivent la même série : on les empile dans une seule colonne
for backtest_col, forecast_col in [
    ("Modéle Prophet Daily", "forecast_prophet_daily"),
    ("Modéle Prophet Weekly", "forecast_prophet_weekly"),
]:
    df_all[backtest_col] = df_all[backtest_col].combine_first(df_all[forecast_col])
    df_all.drop(columns=[forecast_col], inplace=True)

display(df_all)
df_all.to_excel("df_all.xlsx", index=False)


In [ ]:
## sum over 2026

real_pricing = pd.read_gbq("""
SELECT 
DATE_TRUNC(pricing_day, MONTH) AS pricing_month,
    SUM(total_pricing) AS real_monthly_total
FROM `passculture-data-prod.ml_finance_prod.daily_pricing`
GROUP BY pricing_month
ORDER BY pricing_month;  """, project_id="passculture-data-prod")

real_pricing.pricing_month = pd.to_datetime(real_pricing.pricing_month).dt.to_period('M')

# on complète les mois de 2026 non encore réalisés avec les prévisions de df_all
last_real_month = real_pricing.pricing_month.max()
forecast_cols = ["scenario1 DAF", "scenario2 DAF", "Modéle Prophet Daily", "Modéle Prophet Weekly"]
forecast_2026 = df_all[
    (df_all.pricing_month > last_real_month) & (df_all.pricing_month.dt.year == last_real_month.year)
][["pricing_month"] + forecast_cols]

real_pricing = pd.concat([real_pricing, forecast_2026], axis=0, ignore_index=True)
real_pricing = real_pricing[real_pricing.pricing_month.dt.year == last_real_month.year].reset_index(drop=True)

# sur les mois déjà réalisés, chaque colonne reprend le réalisé
for col in forecast_cols:
    real_pricing[col] = real_pricing[col].fillna(real_pricing.real_monthly_total)

real_pricing = real_pricing.set_index(real_pricing.pricing_month.astype(str)).drop(columns=["pricing_month"])
real_pricing.index.name = "mois"
real_pricing.loc[f"Total {last_real_month.year}", forecast_cols] = real_pricing[forecast_cols].sum()

real_pricing.to_excel(f"pricings_{last_real_month.year}.xlsx")
